# Détection de doublons d'observations (ML)\n\nObjectif : prédire si une nouvelle observation est un **doublon** d'une observation déjà saisie aujourd'hui pour le même enfant.\n\nApproche : classification sur des **paires** (duplicate / non-duplicate) avec TF‑IDF + features simples.

In [ ]:
!pip -q install -r requirements.txt

In [ ]:
import pandas as pd\nfrom train import build_example_observations, make_pairs\n\nobs = build_example_observations()\npairs = make_pairs(obs)\npairs

In [ ]:
pairs['label_dup'].value_counts()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import classification_report\nimport numpy as np\n\ndef cosine_sim_sparse(a, b):\n    num = a.multiply(b).sum(axis=1).A.ravel()\n    den = np.linalg.norm(a.toarray(), axis=1) * np.linalg.norm(b.toarray(), axis=1)\n    den = np.where(den == 0, 1e-9, den)\n    return num / den\n\nall_texts = pd.concat([pairs['a_text'], pairs['b_text']], ignore_index=True)\ntfidf = TfidfVectorizer(ngram_range=(1,2), min_df=1)\ntfidf.fit(all_texts)\n\na_vec = tfidf.transform(pairs['a_text'])\nb_vec = tfidf.transform(pairs['b_text'])\nsim = cosine_sim_sparse(a_vec, b_vec)\n\nX = np.column_stack([\n    sim,\n    pairs['same_type'].to_numpy(),\n    pairs['minutes_diff'].to_numpy(),\n    pairs['temp_diff'].to_numpy(),\n])\ny = pairs['label_dup'].to_numpy()\n\nX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)\nmodel = LogisticRegression(max_iter=1000)\nmodel.fit(X_train, y_train)\npred = model.predict(X_test)\nprint(classification_report(y_test, pred, digits=3))

In [ ]:
# Export artefacts\nimport joblib\nfrom pathlib import Path\nimport json\nfrom datetime import datetime\n\nART = Path('artifacts')\nART.mkdir(exist_ok=True)\njoblib.dump(tfidf, ART/'tfidf.pkl')\njoblib.dump(model, ART/'dup_model.pkl')\n\nmeta = {\n  'trained_at': datetime.now().isoformat(timespec='seconds'),\n  'features': ['cosine_sim_tfidf','same_type','minutes_diff','temp_diff'],\n  'model': 'LogisticRegression',\n  'threshold': 0.75\n}\n(ART/'metadata.json').write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')\nmeta